# 🎙️ VoiceBatch Studio v2.0.1 - Android & Desktop Optimized
यह वर्जन डिस्कनेक्शन को रोकने के लिए 'Anti-Sleep' और सभी फिक्स के साथ तैयार है।

In [ ]:
# @title 📥 Step 1: इंस्टॉलेशन फिक्स (All-in-One)
print("⏳ लाइब्रेरी इंस्टॉल हो रही हैं... (torchcodec और coqui-tts फिक्स के साथ)")

# सही पैकेज और वर्जन इंस्टॉल करना
!pip install -q gradio edge-tts librosa soundfile torchcodec
!pip install -q coqui-tts

print("✅ लाइब्रेरी सेटअप पूरा हुआ!")

In [ ]:
# @title 💤 Step 2: Anti-Sleeping Mode (Mobile/Desktop)
import time
from IPython.display import display, Javascript

def anti_sleep():
    display(Javascript('''
        function ClickConnect(){
            console.log("Keeping Colab Active...");
            document.querySelector("colab-connect-button").click() 
        }
        setInterval(ClickConnect, 60000)
    '''))
    print("🚀 Anti-Sleep सक्रिय है! आपका सेशन अब डिस्कनेक्ट नहीं होगा।")

anti_sleep()

In [ ]:
# @title 🚀 Step 3: app.py लिखें और रन करें
import os

app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import asyncio
import edge_tts
import os
import librosa
import soundfile as sf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading XTTS v2 on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def cleanup_audio(audio_path):
    y, sr = librosa.load(audio_path)
    y, _ = librosa.effects.trim(y, top_db=25)
    clean_path = "cleaned_voice.wav"
    sf.write(clean_path, y, sr)
    return clean_path

async def fast_tts(text, voice, speed, pitch):
    output = 'fast_voice.mp3'
    rate = f"{speed:+}%"
    p = f"{pitch:+}Hz"
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=p)
    await communicate.save(output)
    return output

def clone_voice(text, audio_sample, cleanup):
    output_path = 'cloned_voice.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language='hi', file_path=output_path)
    if cleanup: output_path = cleanup_audio(output_path)
    return output_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.1')
    with gr.Tabs():
        with gr.TabItem('🧬 Voice Cloning'):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='टेक्स्ट', lines=5)
                    sample = gr.Audio(label='सैंपल अपलोड करें', type='filepath')
                    use_cleanup = gr.Checkbox(label="Silence Remover & Cleanup", value=True)
                    btn_clone = gr.Button('Clone & Generate', variant='primary')
                output_clone = gr.Audio(label='आउटपुट')
            btn_clone.click(clone_voice, [input_text, sample, use_cleanup], output_clone)
            
        with gr.TabItem('⚡ Standard TTS'):
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='टेक्स्ट', lines=5)
                    v_drop = gr.Dropdown(choices=['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural'], label='आवाज़', value='hi-IN-MadhurNeural')
                    spd = gr.Slider(-50, 50, 0, label="Speed")
                    ptc = gr.Slider(-20, 20, 0, label="Pitch")
                    btn_fast = gr.Button('Generate Fast')
                output_fast = gr.Audio(label='आउटपुट')
            btn_fast.click(lambda t, v, s, p: asyncio.run(fast_tts(t, v, s, p)), [t_text, v_drop, spd, ptc], output_fast)

demo.launch(share=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("🚀 app.py तैयार है! अब ऐप शुरू हो रहा है...")
!python app.py